# 🇪🇨 Análisis de Suicidio en Ecuador (2012–2024)

> **Objetivo:** Explorar la evolución de las tasas de suicidio en Ecuador a nivel nacional y provincial, desagregadas por sexo y grupo etario, usando datos oficiales del INEC.

---

## 📁 Estructura del Proyecto

```
suicidio-ecuador/
├── data/
│   ├── EDG/            # Archivos .sav del Registro de Defunciones Generales (INEC)
│   ├── pobLacion/      # Excel con proyecciones poblacionales quinquenales (INEC)
│   └── mapa/           # Shapefile de provincias del Ecuador
├── notebook/
│   └── Suicidio.ipynb  # Notebook principal
├── outputs/        # Gráficos exportados
├── .gitignore
├── requirements.txt
└── README.md
```

---

## 📋 Fuentes de Datos

| Dataset | Fuente | Período |
|--------|--------|--------|
| Registro de Defunciones Generales | [INEC](https://www.ecuadorencifras.gob.ec/defunciones-generales/) | 2012–2024 |
| Proyecciones Poblacionales Provinciales | [INEC](https://www.ecuadorencifras.gob.ec/proyecciones-poblacionales/) | 2012–2024 |
| Shapefile Provincias Ecuador | INEC / IGM | — |

**Códigos CIE-10 para suicidio:** `X60–X84` *(Lesiones autoinfligidas intencionalmente)*

---

## 1. Configuración del Entorno

### 1.1 Importación de Librerías

In [1]:
# ── Manipulación de datos
import pandas as pd
import glob

# ── Lectura de archivos especializados
import pyreadstat   # Archivos SPSS (.sav)
import openpyxl     # Excel (.xlsx)

# ── Geodatos y visualización
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go

print('✅ Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


---

## 2. Carga y Preparación de Datos

### 2.1 Registro de Defunciones Generales (INEC)

Se cargan los archivos `.sav` de cada año desde `../data/EDG/`.  
Solo se conservan las columnas relevantes para el análisis.

In [2]:
# Buscar y ordenar todos los archivos .sav
archivos = glob.glob('../data/EDG/*.sav')
archivos.sort()
print(f'Archivos encontrados: {len(archivos)}')
for a in archivos:
    print(' ', a)

Archivos encontrados: 13
  ../data/EDG\EDG_2012.sav
  ../data/EDG\EDG_2013.sav
  ../data/EDG\EDG_2014.sav
  ../data/EDG\EDG_2015.sav
  ../data/EDG\EDG_2016.sav
  ../data/EDG\EDG_2017.sav
  ../data/EDG\EDG_2018.sav
  ../data/EDG\EDG_2019.sav
  ../data/EDG\EDG_2020.sav
  ../data/EDG\EDG_2021.sav
  ../data/EDG\EDG_2022.sav
  ../data/EDG\EDG_2023.sav
  ../data/EDG\EDG_2024.sav


In [3]:
# Cargar cada archivo y almacenarlo en una lista
lista_df = []

for archivo in archivos:
    df, meta = pyreadstat.read_sav(archivo)
    lista_df.append(df)
    print(f'  {archivo.split("/")[-1]}  →  {df.shape[0]:,} filas | {df.shape[1]} cols')

  EDG\EDG_2012.sav  →  63,511 filas | 66 cols
  EDG\EDG_2013.sav  →  63,104 filas | 52 cols
  EDG\EDG_2014.sav  →  64,770 filas | 49 cols
  EDG\EDG_2015.sav  →  66,598 filas | 46 cols
  EDG\EDG_2016.sav  →  68,848 filas | 45 cols
  EDG\EDG_2017.sav  →  70,841 filas | 44 cols
  EDG\EDG_2018.sav  →  72,789 filas | 45 cols
  EDG\EDG_2019.sav  →  75,355 filas | 45 cols
  EDG\EDG_2020.sav  →  117,030 filas | 44 cols
  EDG\EDG_2021.sav  →  107,648 filas | 45 cols
  EDG\EDG_2022.sav  →  91,954 filas | 45 cols
  EDG\EDG_2023.sav  →  89,877 filas | 45 cols
  EDG\EDG_2024.sav  →  91,803 filas | 45 cols


In [4]:
# Verificar consistencia de columnas entre años
print('Columnas 2012:', list(lista_df[0].columns))
print('Columnas 2024:', list(lista_df[-1].columns))

Columnas 2012: ['ofi_insc', 'prov_insc', 'cant_insc', 'parr_insc', 'anio_insc', 'mes_insc', 'acta_insc', 'sexo', 'anio_nac', 'mes_nac', 'anio_fall', 'mes_fall', 'edad', 'cod_edad', 'prov_res', 'cant_res', 'parr_res', 'area_res', 'est_civil', 'sabe_leer', 'niv_inst', 'p_etnica', 'lugar_ocur', 'cod_esta', 'prov_fall', 'cant_fall', 'parr_fall', 'area_fall', 'causa4', 'causa', 'mu_matern', 'mor_mat', 'mu_violen', 'lugar_muviolen', 'pro_med', 'cer_por', 'paren_fall', 'tipo_cer', 'causa103', 'cauletra', 'causa67A', 'causa67B', 'causa80', 't1', 'certimed', 'reg_fall', 'residente', 'gedad', 'gedad1', 'gedad2', 'grupoedad', 'dised14', 'gedad3', 'lc1', 'lceti', 'lcnum', 'redad1', 'reg_res', 'gcausa', 'gcausa103', 'codigo', 'grucau11', 'causa667', 'meses', 'dias', 'grupoedad1']
Columnas 2024: ['Numeración', 'prov_insc', 'cant_insc', 'parr_insc', 'anio_insc', 'mes_insc', 'dia_insc', 'fecha_insc', 'nac_fall', 'cod_pais', 'sexo', 'anio_nac', 'mes_nac', 'dia_nac', 'fecha_nac', 'anio_fall', 'mes_fall'

In [5]:
# Seleccionar únicamente las columnas útiles
COLUMNAS_UTILES = [
    'anio_fall',   # Año del fallecimiento
    'sexo',        # Sexo (1=Hombre, 2=Mujer)
    'edad',        # Edad en años
    'cod_edad',    # Código de unidad de edad
    'prov_res',    # Provincia de residencia
    'area_res',    # Área de residencia (urbana/rural)
    'est_civil',   # Estado civil
    'niv_inst',    # Nivel de instrucción
    'causa',       # Causa de muerte (CIE-10, 3 dígitos)
    'causa4',      # Causa de muerte (CIE-10, 4 dígitos)
]

lista_df_reducida = [df[COLUMNAS_UTILES] for df in lista_df]
print(f'✅ Columnas seleccionadas: {len(COLUMNAS_UTILES)}')

✅ Columnas seleccionadas: 10


#### Consolidar todos los años

In [6]:
base_completa = pd.concat(lista_df_reducida, ignore_index=True)
print(f'Base completa: {base_completa.shape[0]:,} registros | {base_completa.shape[1]} columnas')
base_completa.head(3)

Base completa: 1,044,128 registros | 10 columnas


,anio_fall,sexo,edad,cod_edad,prov_res,area_res,est_civil,niv_inst,causa,causa4
0,2012.0,2.0,19.0,4.0,01,2.0,2.0,7.0,O99,O994
1,2012.0,2.0,27.0,4.0,01,1.0,2.0,2.0,O00,O009
2,2012.0,2.0,33.0,4.0,01,2.0,3.0,2.0,O67,O679


### 2.2 Filtrar Casos de Suicidio

Se filtran los registros con códigos **X60–X84** (CIE-10): *Lesiones autoinfligidas intencionalmente*.

In [7]:
print('Tipo de dato causa:', base_completa['causa'].dtype)
print('Muestra de valores:', base_completa['causa'].unique()[:10])

Tipo de dato causa: object
Muestra de valores: ['O99' 'O00' 'O67' 'O74' 'O08' 'O72' 'O96' 'A05' 'C16' 'C25']


In [8]:
# Filtrar suicidios (CIE-10: X60–X84)
mascara = (base_completa['causa'] >= 'X60') & (base_completa['causa'] <= 'X84')
suicidios = base_completa[mascara].copy()

print(f'Casos de suicidio: {len(suicidios):,}')
print(f'Porcentaje del total: {len(suicidios)/len(base_completa)*100:.2f}%')

Casos de suicidio: 14,285
Porcentaje del total: 1.37%


In [9]:
# Distribución por año
suicidios['anio_fall'].value_counts().sort_index()

anio_fall
1955.0       1
1993.0       1
1996.0       1
2001.0       1
2004.0       2
2005.0       1
2006.0       2
2007.0       1
2008.0       2
2009.0       1
2010.0       4
2011.0       3
2012.0     978
2013.0     697
2014.0     754
2015.0    1092
2016.0    1235
2017.0    1202
2018.0    1223
2019.0    1219
2020.0    1165
2021.0    1195
2022.0    1160
2023.0    1202
2024.0    1143
Name: count, dtype: int64

In [10]:
# Restringir al período 2012–2024
suicidios = suicidios[
    (suicidios['anio_fall'] >= 2012) & (suicidios['anio_fall'] <= 2024)
]
print(f'Casos en período 2012–2024: {len(suicidios):,}')

Casos en período 2012–2024: 14,265


### 2.3 Codificar Variables Categóricas

#### Provincia de residencia

In [11]:
MAPA_PROVINCIA = {
    '01': 'azuay',        '02': 'bolivar',       '03': 'cañar',
    '04': 'carchi',       '05': 'cotopaxi',      '06': 'chimborazo',
    '07': 'el_oro',       '08': 'esmeraldas',    '09': 'guayas',
    '10': 'imbabura',     '11': 'loja',          '12': 'los_rios',
    '13': 'manabi',       '14': 'morona',        '15': 'napo',
    '16': 'pastaza',      '17': 'pichincha',     '18': 'tungurahua',
    '19': 'zamora',       '20': 'galapagos',     '21': 'sucumbios',
    '22': 'orellana',     '23': 'santo_domingo', '24': 'santa_elena',
    '88': 'exterior',     '90': 'ignorado'
}

suicidios['provincia'] = suicidios['prov_res'].map(MAPA_PROVINCIA)
print('Distribución por provincia:')
print(suicidios['provincia'].value_counts())

Distribución por provincia:
provincia
pichincha        2752
guayas           2339
azuay            1218
manabi           1038
los_rios          751
tungurahua        747
el_oro            627
cotopaxi          601
chimborazo        545
loja              466
imbabura          438
santo_domingo     400
cañar             329
esmeraldas        303
bolivar           271
sucumbios         256
carchi            227
orellana          220
morona            175
napo              169
santa_elena       147
zamora            122
pastaza            98
galapagos          20
exterior            4
ignorado            2
Name: count, dtype: int64


In [12]:
# Excluir registros sin provincia válida
suicidios = suicidios[~suicidios['provincia'].isin(['exterior', 'ignorado'])]
print(f'Registros tras filtro geográfico: {len(suicidios):,}')

Registros tras filtro geográfico: 14,259


#### Sexo

In [13]:
suicidios['sexo_label'] = suicidios['sexo'].map({1: 'Hombre', 2: 'Mujer'})

conteo = suicidios['sexo_label'].value_counts()
print('Distribución por sexo:')
print(conteo)
print(f'\nRazón Hombre/Mujer: {conteo["Hombre"]/conteo["Mujer"]:.1f}x')

Distribución por sexo:
sexo_label
Hombre    11009
Mujer      3250
Name: count, dtype: int64

Razón Hombre/Mujer: 3.4x


#### Grupos de Edad

In [14]:
BINS   = [0, 14, 19, 29, 44, 64, 120]
LABELS = ['0-14', '15-19', '20-29', '30-44', '45-64', '65+']

suicidios['grupo_edad'] = pd.cut(
    suicidios['edad'], bins=BINS, labels=LABELS, right=True
)

print('Distribución por grupo de edad:')
print(suicidios['grupo_edad'].value_counts().sort_index())

Distribución por grupo de edad:
grupo_edad
0-14      873
15-19    2164
20-29    3961
30-44    3340
45-64    2619
65+      1301
Name: count, dtype: int64


---

### 2.4 Datos de Población (INEC)

Proyecciones poblacionales por provincia, sexo y grupo de edad quinquenal (1990–2035).  
Se recodifican al mismo agrupamiento etario del análisis de suicidio.

In [15]:
PROVINCIAS = [
    'azuay', 'bolivar', 'cañar', 'carchi', 'cotopaxi', 'chimborazo',
    'el_oro', 'esmeraldas', 'guayas', 'imbabura', 'loja', 'los_rios',
    'manabi', 'morona', 'napo', 'pastaza', 'pichincha', 'tungurahua',
    'zamora', 'galapagos', 'sucumbios', 'orellana', 'santo_domingo', 'santa_elena'
]
GENEROS = {'h': 'Hombre', 'm': 'Mujer'}
RUTA_POB = '../data/pobLacion/Tabulado_provincial_edad_quinquenal_1990-2035.xlsx'

lista_pob = []
for prov in PROVINCIAS:
    for sufijo, sexo_label in GENEROS.items():
        hoja = f'{prov}_{sufijo}'
        df_temp = pd.read_excel(RUTA_POB, sheet_name=hoja, header=13)
        df_temp = df_temp.rename(columns={'Unnamed: 1': 'Edad'})
        df_temp = df_temp.drop(columns=['Unnamed: 0'])
        df_temp['provincia'] = prov
        df_temp['sexo']      = sexo_label
        lista_pob.append(df_temp)

poblacion = pd.concat(lista_pob, ignore_index=True)
print(f'Datos de población cargados: {poblacion.shape[0]:,} filas')
poblacion.head(3)

Datos de población cargados: 1,056 filas


,Edad,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2028,2029,2030,2031,2032,2033,2034,2035,provincia,sexo
0,0-4,37068,37507,37930,38276,38510,38561,38427,38179,37831,...,31770,31443,31176,30950,30760,30598,30458,30336,azuay,Hombre
1,5-9,34102,34746,35270,35682,36006,36351,36763,37182,37551,...,34127,33514,32904,32348,31878,31494,31179,30921,azuay,Hombre
2,10-14,30419,31001,31593,32221,32891,33555,34145,34638,35044,...,36224,35752,35274,34779,34248,33682,33089,32496,azuay,Hombre


In [16]:
# Formato ancho → largo
poblacion_larga = poblacion.melt(
    id_vars=['Edad', 'provincia', 'sexo'],
    var_name='anio', value_name='poblacion'
)
poblacion_larga['anio'] = poblacion_larga['anio'].astype(int)
poblacion_larga = poblacion_larga[
    (poblacion_larga['anio'] >= 2012) & (poblacion_larga['anio'] <= 2024)
]
print(f'Registros tras filtro temporal: {poblacion_larga.shape[0]:,}')

Registros tras filtro temporal: 13,728


In [17]:
# Recodificar grupos quinquenales al esquema del análisis
MAPA_EDAD = {
    '0-4': '0-14',  '5-9': '0-14',  '10-14': '0-14',
    '15-19': '15-19',
    '20-24': '20-29', '25-29': '20-29',
    '30-34': '30-44', '35-39': '30-44', '40-44': '30-44',
    '45-49': '45-64', '50-54': '45-64', '55-59': '45-64', '60-64': '45-64',
    '65-69': '65+',   '70-74': '65+',   '75-79': '65+',   '80-84': '65+',
    '85-89': '65+',   '90-94': '65+',   '95-99': '65+',   '100 y más': '65+'
}

poblacion_larga = poblacion_larga[poblacion_larga['Edad'] != 'Total']
poblacion_larga['grupo_edad'] = poblacion_larga['Edad'].map(MAPA_EDAD)

sin_mapear = poblacion_larga['grupo_edad'].isna().sum()
print(f'Grupos sin mapear: {sin_mapear}  (debe ser 0)')

Grupos sin mapear: 0  (debe ser 0)


In [18]:
# Agregar por grupos ampliados
poblacion_agrupada = poblacion_larga.groupby(
    ['anio', 'provincia', 'sexo', 'grupo_edad'], as_index=False
)['poblacion'].sum()

print(f'Base de población agrupada: {poblacion_agrupada.shape[0]:,} filas')
poblacion_agrupada.head(3)

Base de población agrupada: 3,744 filas


,anio,provincia,sexo,grupo_edad,poblacion
0,2012,azuay,Hombre,0-14,109798
1,2012,azuay,Hombre,15-19,37086
2,2012,azuay,Hombre,20-29,67688


---

### 2.5 Construcción de la Base Final

Se combina la base de suicidios con la de población para calcular **tasas por 100,000 hab.**

In [19]:
# Agregar casos por año, provincia, sexo y grupo de edad
casos_agrupados = suicidios.groupby(
    ['anio_fall', 'provincia', 'sexo_label', 'grupo_edad'], as_index=False
).size().rename(columns={
    'size': 'casos', 'anio_fall': 'anio', 'sexo_label': 'sexo'
})
casos_agrupados['anio'] = casos_agrupados['anio'].astype(int)

print(f'Combinaciones con casos registrados: {len(casos_agrupados):,}')
casos_agrupados.head(3)

Combinaciones con casos registrados: 3,744


C:\Users\ASUS\AppData\Local\Temp\ipykernel_1768\2513218698.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  casos_agrupados = suicidios.groupby(


,anio,provincia,sexo,grupo_edad,casos
0,2012,azuay,Hombre,0-14,3
1,2012,azuay,Hombre,15-19,11
2,2012,azuay,Hombre,20-29,18


In [20]:
# Merge con población (left join)
base_final = poblacion_agrupada.merge(
    casos_agrupados,
    on=['anio', 'provincia', 'sexo', 'grupo_edad'],
    how='left'
)
base_final['casos'] = base_final['casos'].fillna(0)

print(f'Base final: {base_final.shape[0]:,} filas | {base_final.shape[1]} columnas')
base_final.head(3)

Base final: 3,744 filas | 6 columnas


,anio,provincia,sexo,grupo_edad,poblacion,casos
0,2012,azuay,Hombre,0-14,109798,3
1,2012,azuay,Hombre,15-19,37086,11
2,2012,azuay,Hombre,20-29,67688,18


In [21]:
# Calcular tasa por 100,000 habitantes
base_final['tasa_100k'] = (base_final['casos'] / base_final['poblacion']) * 100_000

print('Top 10 estratos con mayor tasa:')
base_final.sort_values('tasa_100k', ascending=False).head(10)

Top 10 estratos con mayor tasa:


,anio,provincia,sexo,grupo_edad,poblacion,casos,tasa_100k
961,2015,galapagos,Hombre,15-19,1138,2,175.746924
2977,2022,galapagos,Hombre,15-19,1167,2,171.379606
1189,2016,cañar,Hombre,15-19,10787,10,92.704181
1429,2016,zamora,Hombre,15-19,5971,5,83.738067
1255,2016,galapagos,Mujer,15-19,1209,1,82.712986
175,2012,morona,Mujer,15-19,9676,7,72.343944
2692,2021,galapagos,Hombre,45-64,2826,2,70.771408
2485,2020,napo,Hombre,15-19,7072,5,70.701357
1046,2015,napo,Hombre,20-29,11953,8,66.928804
305,2013,bolivar,Hombre,65+,9028,6,66.459903


---

## 3. Estadísticas Descriptivas

Resumen general del período analizado antes de pasar a las visualizaciones.

In [22]:
# ── Tabla resumen por año ─────────────────────────────────────────────────
resumen_anual = (
    base_final.groupby('anio')
    .apply(lambda d: pd.Series({
        'casos'     : int(d['casos'].sum()),
        'poblacion' : int(d['poblacion'].sum()),
        'tasa_100k' : round((d['casos'].sum() / d['poblacion'].sum()) * 100_000, 2)
    }))
    .reset_index()
)

resumen_anual.columns = ['Año', 'Casos', 'Población', 'Tasa / 100k']
resumen_anual['Casos']     = resumen_anual['Casos'].apply(lambda x: f'{x:,}')
resumen_anual['Población'] = resumen_anual['Población'].apply(lambda x: f'{x:,}')
resumen_anual

C:\Users\ASUS\AppData\Local\Temp\ipykernel_1768\1134241960.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: pd.Series({


,Año,Casos,Población,Tasa / 100k
0,2012,977.0,"15,440,550.0",6.33
1,2013,697.0,"15,717,514.0",4.43
2,2014,753.0,"15,994,366.0",4.71
3,2015,"1,091.0","16,269,171.0",6.71
4,2016,"1,235.0","16,532,265.0",7.47
5,2017,"1,202.0","16,792,803.0",7.16
6,2018,"1,221.0","17,077,883.0",7.15
7,2019,"1,217.0","17,356,880.0",7.01
8,2020,"1,165.0","17,526,455.0",6.65
9,2021,"1,195.0","17,614,396.0",6.78


#### Tasa de Suicidio por Nivel de Instrucción

La variable `niv_inst` codifica el nivel educativo del fallecido según el INEC.  
Se calcula el porcentaje de casos en cada categoría (no tasa por 100k, ya que no tenemos
el denominador poblacional desagregado por instrucción).

In [23]:
# Diccionario de etiquetas INEC para niv_inst
MAPA_INSTRUCCION = {
    1: 'Ninguno',
    2: 'Centro de alfabetización',
    3: 'Primaria',
    4: 'Secundaria',
    5: 'Superior universitaria',
    6: 'Superior no universitaria',
    7: 'Posgrado',
    9: 'Ignorado / Sin dato'
}

suicidios['instruccion'] = suicidios['niv_inst'].map(MAPA_INSTRUCCION).fillna('Ignorado / Sin dato')

conteo_inst = (
    suicidios['instruccion']
    .value_counts()
    .reset_index()
)
conteo_inst.columns = ['Nivel de instrucción', 'Casos']
conteo_inst['%'] = (conteo_inst['Casos'] / conteo_inst['Casos'].sum() * 100).round(1)
print(conteo_inst.to_string(index=False))

     Nivel de instrucción  Casos    %
                 Primaria   3666 25.7
 Centro de alfabetización   3211 22.5
   Superior universitaria   2389 16.8
               Secundaria   2288 16.0
      Ignorado / Sin dato   1565 11.0
                 Posgrado    871  6.1
Superior no universitaria    178  1.2
                  Ninguno     91  0.6


In [24]:
import plotly.express as px

# Excluir 'Ignorado' para la visualización
inst_plot = conteo_inst[conteo_inst['Nivel de instrucción'] != 'Ignorado / Sin dato'].copy()
inst_plot = inst_plot.sort_values('Casos')

fig_inst = px.bar(
    inst_plot,
    x='Casos',
    y='Nivel de instrucción',
    orientation='h',
    text='%',
    color='Casos',
    color_continuous_scale='Reds',
    title='<b>Casos de suicidio por nivel de instrucción</b><br>Ecuador, 2012–2024',
    labels={'Casos': 'Número de casos', 'Nivel de instrucción': ''},
)

fig_inst.update_traces(texttemplate='%{text}%', textposition='outside')
fig_inst.update_layout(
    showlegend=False,
    coloraxis_showscale=False,
    xaxis_title='Número de casos',
    margin=dict(l=180, r=60, t=80, b=40),
    height=400
)
fig_inst.show()

In [25]:
# Evolución temporal por nivel de instrucción (top 4 niveles)
TOP_NIVELES = ['Primaria', 'Secundaria', 'Superior universitaria', 'Ninguno']

inst_anual = (
    suicidios[suicidios['instruccion'].isin(TOP_NIVELES)]
    .groupby(['anio_fall', 'instruccion'])
    .size()
    .reset_index(name='casos')
)

fig_inst_tiempo = px.line(
    inst_anual,
    x='anio_fall',
    y='casos',
    color='instruccion',
    markers=True,
    title='<b>Evolución de casos por nivel de instrucción</b><br>Top 4 niveles · Ecuador, 2012–2024',
    labels={'anio_fall': 'Año', 'casos': 'Casos', 'instruccion': 'Nivel de instrucción'},
)
fig_inst_tiempo.update_layout(xaxis=dict(dtick=1), hovermode='x unified')
fig_inst_tiempo.show()

#### 5. Pirámide de Casos por Sexo y Grupo de Edad

Visualiza la distribución de los casos de suicidio según sexo y grupo etario,
acumulada para todo el período 2012–2024.

In [26]:
import plotly.graph_objects as go

# Agregar casos por sexo y grupo de edad (todo el período)
piramide = (
    suicidios.groupby(['grupo_edad', 'sexo_label'])
    .size()
    .reset_index(name='casos')
)

ORDEN_EDAD = ['0-14', '15-19', '20-29', '30-44', '45-64', '65+']
piramide['grupo_edad'] = pd.Categorical(piramide['grupo_edad'], categories=ORDEN_EDAD, ordered=True)
piramide = piramide.sort_values('grupo_edad')

hombres = piramide[piramide['sexo_label'] == 'Hombre']['casos'].values
mujeres = piramide[piramide['sexo_label'] == 'Mujer']['casos'].values

fig_piramide = go.Figure()

# Hombres → lado izquierdo (valores negativos)
fig_piramide.add_trace(go.Bar(
    y=ORDEN_EDAD,
    x=-hombres,
    name='Hombre',
    orientation='h',
    marker_color='steelblue',
    text=hombres,
    textposition='outside',
))

# Mujeres → lado derecho
fig_piramide.add_trace(go.Bar(
    y=ORDEN_EDAD,
    x=mujeres,
    name='Mujer',
    orientation='h',
    marker_color='salmon',
    text=mujeres,
    textposition='outside',
))

# Eje X con etiquetas absolutas
max_val = max(hombres.max(), mujeres.max())
ticks   = list(range(-int(max_val) - 100, int(max_val) + 200, 200))

fig_piramide.update_layout(
    title='<b>Pirámide de casos de suicidio por sexo y grupo de edad</b><br>Ecuador, 2012–2024',
    barmode='overlay',
    bargap=0.15,
    xaxis=dict(
        tickvals=ticks,
        ticktext=[str(abs(t)) for t in ticks],
        title='Número de casos',
    ),
    yaxis_title='Grupo de edad',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=450,
)
fig_piramide.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_1768\632406138.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  suicidios.groupby(['grupo_edad', 'sexo_label'])


In [27]:
# Tabla resumen de la pirámide
resumen_piramide = piramide.pivot(index='grupo_edad', columns='sexo_label', values='casos').fillna(0).astype(int)
resumen_piramide['Total']    = resumen_piramide.sum(axis=1)
resumen_piramide['Razón H/M'] = (resumen_piramide['Hombre'] / resumen_piramide['Mujer']).round(1)

print('Casos por grupo de edad y sexo (2012–2024):')
print(resumen_piramide.to_string())

Casos por grupo de edad y sexo (2012–2024):
sexo_label  Hombre  Mujer  Total  Razón H/M
grupo_edad                                 
0-14           491    382    873        1.3
15-19         1337    827   2164        1.6
20-29         3129    832   3961        3.8
30-44         2740    600   3340        4.6
45-64         2204    415   2619        5.3
65+           1107    194   1301        5.7


---

## 3. Visualización

### 3.1 Series Temporales

In [28]:
# Serie por provincia
serie_provincia = base_final.groupby(['anio', 'provincia'], as_index=False).agg(
    casos     = ('casos',    'sum'),
    poblacion = ('poblacion', 'sum')
)
serie_provincia['tasa_100k'] = (
    serie_provincia['casos'] / serie_provincia['poblacion']
) * 100_000

# Serie nacional
serie_nacional = base_final.groupby('anio', as_index=False).agg(
    casos     = ('casos',    'sum'),
    poblacion = ('poblacion', 'sum')
)
serie_nacional['tasa_100k'] = (
    serie_nacional['casos'] / serie_nacional['poblacion']
) * 100_000
serie_nacional['provincia'] = 'Ecuador (nacional)'

print('Tasa nacional por año:')
print(serie_nacional[['anio', 'tasa_100k']].to_string(index=False))

Tasa nacional por año:
 anio  tasa_100k
 2012   6.327495
 2013   4.434544
 2014   4.707908
 2015   6.705935
 2016   7.470241
 2017   7.157828
 2018   7.149598
 2019   7.011629
 2020   6.647094
 2021   6.784224
 2022   6.548012
 2023   6.739621
 2024   6.361814


### 3.2 Mapa Coroplético — Tasa Promedio por Provincia

In [29]:
# Cargar shapefile
provincias_shp = gpd.read_file('../data/mapa/provincias_ecuador.shp')
print('Columnas del shapefile:', provincias_shp.columns.tolist())

Columnas del shapefile: ['nom_pro', 'PROVINCIA', 'geometry']


In [30]:
# Estandarizar nombres para el merge
MAPA_SHP = {
    'azuay': 'AZUAY',             'bolivar': 'BOLIVAR',
    'cañar': 'CAÑAR',             'carchi': 'CARCHI',
    'cotopaxi': 'COTOPAXI',       'chimborazo': 'CHIMBORAZO',
    'el_oro': 'EL ORO',           'esmeraldas': 'ESMERALDAS',
    'guayas': 'GUAYAS',           'imbabura': 'IMBABURA',
    'loja': 'LOJA',               'los_rios': 'LOS RIOS',
    'manabi': 'MANABI',           'morona': 'MORONA SANTIAGO',
    'napo': 'NAPO',               'pastaza': 'PASTAZA',
    'pichincha': 'PICHINCHA',     'tungurahua': 'TUNGURAHUA',
    'zamora': 'ZAMORA CHINCHIPE', 'galapagos': 'GALAPAGOS',
    'sucumbios': 'SUCUMBIOS',     'orellana': 'ORELLANA',
    'santo_domingo': 'SANTO DOMINGO DE LOS TSACHILAS',
    'santa_elena': 'SANTA ELENA'
}

mapa_datos = serie_provincia.groupby('provincia', as_index=False)['tasa_100k'].mean()
mapa_datos['provincia_shp'] = mapa_datos['provincia'].map(MAPA_SHP)
print(f'Provincias sin mapear: {mapa_datos["provincia_shp"].isna().sum()}')

Provincias sin mapear: 0


In [31]:
# Unir con geometrías
mapa_completo = provincias_shp.merge(
    mapa_datos, left_on='PROVINCIA', right_on='provincia_shp'
)
mapa_completo = mapa_completo.to_crs(epsg=4326)
mapa_completo['geometry'] = mapa_completo['geometry'].simplify(
    tolerance=0.01, preserve_topology=True
)
print(f'Provincias en el mapa: {len(mapa_completo)}')

Provincias en el mapa: 24


In [32]:
# Mapa interactivo
fig_mapa = px.choropleth(
    mapa_completo,
    geojson=mapa_completo.geometry,
    locations=mapa_completo.index,
    color='tasa_100k',
    color_continuous_scale='Reds',
    title='<b>Tasa promedio de suicidio por 100,000 hab.</b><br>Ecuador, 2012–2024',
    hover_name='provincia_shp',
    hover_data={'tasa_100k': ':.2f'},
    labels={'tasa_100k': 'Tasa / 100k'},
)
fig_mapa.update_geos(fitbounds='locations', visible=False)
fig_mapa.update_layout(
    margin={'r': 0, 't': 60, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(title='Tasa / 100k')
)
fig_mapa.show()

### 3.3 Serie Temporal por Provincia

In [33]:
DESTACADAS = ['loja', 'Ecuador (nacional)']

serie_grafico = pd.concat([
    serie_provincia[['anio', 'provincia', 'tasa_100k']],
    serie_nacional[['anio',  'provincia', 'tasa_100k']]
], ignore_index=True)

fig = px.line(
    serie_grafico,
    x='anio', y='tasa_100k', color='provincia',
    title='<b>Tasa de suicidio por 100,000 hab.</b><br>Ecuador por provincia, 2012–2024',
    labels={'anio': 'Año', 'tasa_100k': 'Tasa / 100,000 hab.', 'provincia': 'Provincia'},
)

for trace in fig.data:
    if trace.name in DESTACADAS:
        trace.line.width = 3.5
        trace.opacity   = 1.0
    else:
        trace.line.width = 1.0
        trace.opacity   = 0.20

fig.update_layout(
    xaxis=dict(dtick=1),
    hovermode='x unified',
    legend_title_text='Provincia'
)
fig.show()

---

## 4. Conclusiones y Limitaciones

### Hallazgos principales

- *(Completa esta sección con los resultados de tu análisis)*

---

### Limitaciones

| Limitación | Descripción |
|-----------|-------------|
| **Subregistro** | Las defunciones clasificadas como “causas indeterminadas” pueden incluir suicidios no reportados. |
| **Proyecciones de población** | Los denominadores son proyecciones del INEC, no censos reales para cada año. |
| **Período intercensal** | Las proyecciones acumulan error en años alejados del último censo. |
| **Desagregación geográfica** | El análisis es a nivel provincial; no se puede desagregar a cantón o parroquia. |

---

*Análisis elaborado con datos públicos del INEC | Ecuador*